# MERRA-2 VPD Statistics Processing

This notebook processes hourly MERRA-2 VPD (Vapor Pressure Deficit) data to compute daily afternoon VPD statistics.

**Afternoon window:** 12pm-6pm UTC (hours 12-17)  
**Temporal resolution:** Daily  
**Spatial resolution:** Grid cell level (lat, lon preserved)  
**Spatial extent:** US states only (masked using region_mask.nc)  
**Time period:** 1984-2025

## Metrics Computed Per Day

For each day, we compute VPD statistics across the afternoon window:

### VPD Metrics
1. `vpd_mean` - Mean VPD during afternoon window (kPa)
2. `vpd_min` - Minimum VPD during afternoon window (kPa)
3. `vpd_max` - Maximum VPD during afternoon window (kPa)

## Output

- **Directory:** `processed_vpd/`
- **File pattern:** `vpd_{YYYYMM}.nc`
- **Format:** NetCDF4 with dimensions (time, lat, lon)
- **Variables:** 3 metrics (daily VPD statistics)
- **Fill value:** NaN for non-US areas and missing data

In [ ]:
# Standard imports
import xarray as xr
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime, timedelta
from tqdm.notebook import tqdm
import os
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Configuration
INPUT_DIR = Path('../daily_data') if Path('../daily_data').exists() else Path('daily_data')
OUTPUT_DIR = Path('processed_vpd')
OUTPUT_DIR.mkdir(exist_ok=True)

# Afternoon hours (12pm-6pm UTC): hours 12-17
AFTERNOON_HOURS = list(range(12, 18))

# Date range
START_YEAR = 1984
END_YEAR = 2025

print(f"Input directory: {INPUT_DIR.resolve()}")
print(f"Output directory: {OUTPUT_DIR.resolve()}")

In [ ]:
def compute_daily_vpd_stats(date):
    """
    Compute afternoon VPD statistics for a single day.
    
    Parameters
    ----------
    date : datetime
        Date to process
    
    Returns
    -------
    dict or None
        Dictionary with metric names as keys and 2D arrays (lat, lon) as values,
        or None if file is missing
    """
    date_str = date.strftime('%Y%m%d')
    file_path = INPUT_DIR / f'merra2_us_{date_str}.nc'
    
    if not file_path.exists():
        return None
    
    try:
        ds = xr.open_dataset(file_path)
        # Extract afternoon hours (12pm-6pm = hours 12-17)
        vpd = ds.VPD.isel(time=AFTERNOON_HOURS).values  # shape: (6 hours, lat, lon)
        ds.close()
        
        # Clamp negative VPD values to 0 (VPD cannot be negative by definition)
        # Small negative values can occur due to numerical precision in the calculation
        vpd = np.maximum(vpd, 0.0)
        
        # Compute statistics across afternoon hours
        # Compute over time dimension (axis=0) to get daily stats per grid cell
        stats = {
            'vpd_mean': np.nanmean(vpd, axis=0).astype(np.float32),
            'vpd_min': np.nanmin(vpd, axis=0).astype(np.float32),
            'vpd_max': np.nanmax(vpd, axis=0).astype(np.float32),
        }
        
        # Validation: check for extremely high VPD values
        # VPD typically < 10 kPa, though extreme desert conditions can be higher
        for metric, values in stats.items():
            max_val = np.nanmax(values)
            if max_val > 15.0:  # Warning threshold for extremely high VPD
                print(
                    f"Warning: Very high VPD detected for {metric} on {date_str}: "
                    f"max value = {max_val:.2f} kPa (typical range: 0-10 kPa)"
                )
        
        return stats
        
    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return None

In [ ]:
def process_month(year, month):
    """
    Process all days in a given month and save to NetCDF.
    
    Parameters
    ----------
    year : int
        Year to process
    month : int
        Month to process (1-12)
    
    Returns
    -------
    bool
        True if successful, False otherwise
    """
    output_file = OUTPUT_DIR / f'vpd_{year}{month:02d}.nc'
    
    # Skip if output file already exists
    if os.path.exists(output_file):
        print(f"Output file {output_file.name} already exists. Skipping.")
        return True
    
    # Generate all dates for the month
    start_date = datetime(year, month, 1)
    if month == 12:
        end_date = datetime(year + 1, 1, 1) - timedelta(days=1)
    else:
        end_date = datetime(year, month + 1, 1) - timedelta(days=1)
    
    dates = pd.date_range(start_date, end_date, freq='D')
    n_days = len(dates)
    
    # Get lat/lon coordinates from first available file
    lat_coords = None
    lon_coords = None
    for date in dates:
        date_str = date.strftime('%Y%m%d')
        file_path = INPUT_DIR / f'merra2_us_{date_str}.nc'
        if file_path.exists():
            ds = xr.open_dataset(file_path)
            lat_coords = ds.lat.values
            lon_coords = ds.lon.values
            ds.close()
            break
    
    if lat_coords is None:
        print(f"No valid input files found for {year}-{month:02d}")
        return False
    
    n_lat = len(lat_coords)
    n_lon = len(lon_coords)
    
    # Load US state mask
    mask_path = Path('masks/region_mask.nc')
    if not mask_path.exists():
        print(f"Warning: Region mask file not found at {mask_path}")
        print("Processing without spatial masking.")
        us_mask = None
    else:
        try:
            ds_mask = xr.open_dataset(mask_path)
            state_mask = ds_mask.state_mask.values
            ds_mask.close()
            
            # Create boolean mask: True for US states (state_mask > 0), False otherwise
            us_mask = state_mask > 0
            
            # Verify mask dimensions match data dimensions
            if us_mask.shape != (n_lat, n_lon):
                print(f"Warning: Mask dimensions {us_mask.shape} don't match data dimensions ({n_lat}, {n_lon})")
                print("Processing without spatial masking.")
                us_mask = None
        except Exception as e:
            print(f"Warning: Error loading region mask: {e}")
            print("Processing without spatial masking.")
            us_mask = None
    
    # Initialize arrays for daily statistics
    metric_names = [
        'vpd_mean',
        'vpd_min',
        'vpd_max',
    ]
    
    daily_data = {metric: np.full((n_days, n_lat, n_lon), np.nan, dtype=np.float32) 
                  for metric in metric_names}
    
    # Process each day
    print(f"Processing {year}-{month:02d}: {n_days} days")
    for i, date in enumerate(tqdm(dates, desc=f"{year}-{month:02d}")):
        day_stats = compute_daily_vpd_stats(date)
        if day_stats is not None:
            for metric in metric_names:
                daily_data[metric][i, :, :] = day_stats[metric]
                
                # Apply US mask: set non-US grid cells to NaN
                if us_mask is not None:
                    daily_data[metric][i, ~us_mask] = np.nan
    
    # Create xarray Dataset for daily data
    ds_daily = xr.Dataset(
        data_vars={metric: (['time', 'lat', 'lon'], daily_data[metric]) 
                   for metric in metric_names},
        coords={
            'time': dates,
            'lat': lat_coords,
            'lon': lon_coords,
        }
    )
    
    # Add metadata
    ds_daily.attrs['title'] = 'MERRA-2 VPD Statistics'
    ds_daily.attrs['description'] = 'Daily afternoon (12pm-6pm UTC) VPD statistics'
    ds_daily.attrs['source'] = 'MERRA-2 M2T1NXSLV collection'
    ds_daily.attrs['afternoon_hours'] = '12-17 (UTC)'
    ds_daily.attrs['temporal_resolution'] = 'Daily'
    ds_daily.attrs['spatial_mask'] = 'US states only (state_mask > 0)'
    ds_daily.attrs['created'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    
    # Add variable metadata
    metric_descriptions = {
        'vpd_mean': ('Mean VPD during afternoon window', 
                    'Daily mean vapor pressure deficit during afternoon hours (12pm-6pm UTC)'),
        'vpd_min': ('Minimum VPD during afternoon window', 
                   'Daily minimum vapor pressure deficit during afternoon hours (12pm-6pm UTC)'),
        'vpd_max': ('Maximum VPD during afternoon window', 
                   'Daily maximum vapor pressure deficit during afternoon hours (12pm-6pm UTC)'),
    }
    
    for metric, (long_name, description) in metric_descriptions.items():
        ds_daily[metric].attrs.update({
            'long_name': long_name,
            'units': 'kPa',
            'description': description
        })
    
    # Save to NetCDF with compression
    encoding = {var: {'zlib': True, 'complevel': 4, 'dtype': 'float32'} 
                for var in ds_daily.data_vars}
    ds_daily.to_netcdf(output_file, encoding=encoding)
    ds_daily.close()
    
    print(f"Saved: {output_file.name}")
    return True

## Test Single Month

Test processing for a single month to verify the workflow.

In [ ]:
# Test with a recent month
process_month(2023, 6)

## Process All Months (1984-2025)

Process all months in the dataset. This will skip any months that have already been processed.

In [ ]:
# Process all years and months
for year in range(START_YEAR, END_YEAR + 1):
    for month in range(1, 13):
        try:
            process_month(year, month)
        except Exception as e:
            print(f"Error processing {year}-{month:02d}: {e}")
            continue

## Verify Output

Quick verification of the output files.

In [ ]:
# List all output files
output_files = sorted(OUTPUT_DIR.glob('*.nc'))
print(f"Total output files: {len(output_files)}\n")

if output_files:
    # Inspect most recent file
    latest_file = output_files[-1]
    print(f"Inspecting: {latest_file.name}")
    ds = xr.open_dataset(latest_file)
    print(ds)
    print("\nData variables:")
    print(list(ds.data_vars))
    ds.close()